In [5]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'],
                               config['paths']['processed_file'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")


In [3]:
df = pd.read_parquet(PROCESSED_PATH)
print(f"Total rows: {len(df):,}")

features = ["cpu", "memory", "duration"]
target   = "energy_kwh"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Total rows: 19,523,808
Train: 15,619,046 | Test: 3,904,762


In [6]:
# ============================================================
# 데이터 로드 + 샘플링 + 증강
# ============================================================
df_full = pd.read_parquet(PROCESSED_PATH)
print(f"Original rows: {len(df_full):,}")
print(f"Columns: {df_full.columns.tolist()}")

SAMPLE_SIZE = 2_000_000
df = df_full.sample(n=SAMPLE_SIZE, random_state=config['model']['random_state'])
del df_full; gc.collect()

DURATION_MULTIPLIERS = [1, 2, 4, 6, 12]

augmented_dfs = []
for m in DURATION_MULTIPLIERS:
    df_aug = df.copy()
    df_aug['duration']  = df_aug['duration'] * m
    df_aug['energy_kwh'] = df_aug['energy_kwh'] * m
    # power_w는 duration과 무관하므로 그대로 유지
    augmented_dfs.append(df_aug)

df_aug = pd.concat(augmented_dfs, ignore_index=True)
del augmented_dfs; gc.collect()
print(f"Augmented rows: {len(df_aug):,}")

Original rows: 19,523,808
Columns: ['machine_id', 'start_time_sec', 'end_time_sec', 'duration', 'hour', 'cpu', 'memory', 'power_w', 'energy_kwh']
Augmented rows: 10,000,000


In [8]:
# ============================================================
# 피처 설정 (power_w 추가!)
# power_w = 200 + cpu*300 + memory*50 (이미 전처리에서 계산됨)
# energy_kwh = power_w * duration / 3600 / 1000 (거의 선형 관계)
# ============================================================
features = ["power_w", "duration"]  # cpu/memory 대신 power_w 사용
target = "energy_kwh"

X = df_aug[features]
y = df_aug[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)
del df_aug; gc.collect()
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 8,000,000 | Test: 2,000,000


In [9]:
# ============================================================
# RF + LR 학습 (power_w 추가)
# ============================================================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
lr = LinearRegression()

rf.fit(X_train, y_train)
lr.fit(X_train, y_train)
print("Base models trained!")


Base models trained!


In [10]:
# ============================================================
# Stacking - OOF 예측 (cv=5) (power_w 추가)
# ============================================================
rf_oof = cross_val_predict(rf, X_train, y_train, cv=5)
lr_oof = cross_val_predict(lr, X_train, y_train, cv=5)

meta_train = np.column_stack([rf_oof, lr_oof])
meta_model = LinearRegression()
meta_model.fit(meta_train, y_train)
print("Meta model trained!")
print(f"  RF coef : {meta_model.coef_[0]:.4f}")
print(f"  LR coef : {meta_model.coef_[1]:.4f}")

# ============================================================
# 평가
# ============================================================
rf_pred    = rf.predict(X_test)
lr_pred    = lr.predict(X_test)
meta_test  = np.column_stack([rf_pred, lr_pred])
stack_pred = meta_model.predict(meta_test)

rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
mae  = mean_absolute_error(y_test, stack_pred)
r2   = r2_score(y_test, stack_pred)
mape = np.mean(np.abs((y_test - stack_pred) / y_test)) * 100

print(f"\n[Stacking 최종 성능 (power_w + duration)]")
print(f"  RMSE : {rmse:.8f}")
print(f"  MAE  : {mae:.8f}")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")

# ============================================================
# 모델 저장
# ============================================================
rf_file   = os.path.join(MODEL_PATH, config['model']['model_names']['randomforest'])
lr_file   = os.path.join(MODEL_PATH, config['model']['model_names']['linearregression'])
meta_file = os.path.join(MODEL_PATH, config['model']['model_names']['stacking_meta'])

with open(rf_file, 'wb') as f:
    pickle.dump(rf, f)
print(f"\nSaved RF   : {rf_file} ({os.path.getsize(rf_file)/(1024**2):.1f} MB)")

with open(lr_file, 'wb') as f:
    pickle.dump(lr, f)
print(f"Saved LR   : {lr_file} ({os.path.getsize(lr_file)/(1024**2):.1f} MB)")

with open(meta_file, 'wb') as f:
    pickle.dump(meta_model, f)
print(f"Saved Meta : {meta_file} ({os.path.getsize(meta_file)/(1024**2):.1f} MB)")

# ============================================================
# phase1_full_results.json 업데이트
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json'])
with open(result_file, 'r') as f:
    results_json = json.load(f)

results_json['stacking'] = {
    'method'      : 'Stacking (RF + LR) with power_w feature + Data Augmentation',
    'base_models' : ['RandomForest', 'LinearRegression'],
    'meta_model'  : 'LinearRegression',
    'rf_params'   : {
        'n_estimators'     : 100,
        'max_depth'        : 15,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
    },
    'augmentation': {
        'duration_multipliers': DURATION_MULTIPLIERS,
        'sample_size'         : SAMPLE_SIZE,
        'augmented_rows'      : SAMPLE_SIZE * len(DURATION_MULTIPLIERS),
    },
    'rf_coef'     : float(meta_model.coef_[0]),
    'lr_coef'     : float(meta_model.coef_[1]),
    'features'    : features,
    'rmse'        : float(rmse),
    'mae'         : float(mae),
    'r2'          : float(r2),
    'mape'        : float(mape),
}
results_json['best_model'] = 'Stacking'
results_json['features']   = features

with open(result_file, 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f"\nResults updated: {result_file}")
print("Done!")

Meta model trained!
  RF coef : 0.9985
  LR coef : 0.0015

[Stacking 최종 성능 (power_w + duration)]
  RMSE : 0.00003623
  MAE  : 0.00000779
  R2   : 0.99999970
  MAPE : 0.0834%

Saved RF   : /content/drive/MyDrive/EcoTracing/models/energy_model_randomforest.pkl (234.4 MB)
Saved LR   : /content/drive/MyDrive/EcoTracing/models/energy_model_linearregression.pkl (0.0 MB)
Saved Meta : /content/drive/MyDrive/EcoTracing/models/energy_model_stacking_meta.pkl (0.0 MB)

Results updated: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results.json
Done!


In [ ]:
# ============================================================
# 베이스 모델 학습 (기존 피처)
# ============================================================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
lr = LinearRegression()

rf.fit(X_train, y_train)
lr.fit(X_train, y_train)
print("Base models trained!")

# 베이스 예측
rf_pred_test = rf.predict(X_test)
lr_pred_test = lr.predict(X_test)

Base models trained!


In [5]:
# ============================================================
# Method 1: Weighted Blending
# 교차검증으로 각 모델 RMSE 계산 후 역수로 가중치 결정
# ============================================================
rf_cv_pred = cross_val_predict(rf, X_train, y_train, cv=3)
lr_cv_pred = cross_val_predict(lr, X_train, y_train, cv=3)

rf_cv_rmse = np.sqrt(mean_squared_error(y_train, rf_cv_pred))
lr_cv_rmse = np.sqrt(mean_squared_error(y_train, lr_cv_pred))

# 오차가 낮을수록 가중치 높게 (역수 정규화)
rf_weight = (1 / rf_cv_rmse) / ((1 / rf_cv_rmse) + (1 / lr_cv_rmse))
lr_weight = (1 / lr_cv_rmse) / ((1 / rf_cv_rmse) + (1 / lr_cv_rmse))

print(f"\n[Blending 가중치]")
print(f"  RF weight : {rf_weight:.4f}")
print(f"  LR weight : {lr_weight:.4f}")

blend_pred = rf_weight * rf_pred_test + lr_weight * lr_pred_test

blend_rmse = np.sqrt(mean_squared_error(y_test, blend_pred))
blend_mae  = mean_absolute_error(y_test, blend_pred)
blend_r2   = r2_score(y_test, blend_pred)
blend_mape = np.mean(np.abs((y_test - blend_pred) / y_test)) * 100

print(f"\n[Method 1: Weighted Blending]")
print(f"  RMSE : {blend_rmse:.8f}")
print(f"  MAE  : {blend_mae:.8f}")
print(f"  R2   : {blend_r2:.8f}")
print(f"  MAPE : {blend_mape:.4f}%")


[Blending 가중치]
  RF weight : 0.8245
  LR weight : 0.1755

[Method 1: Weighted Blending]
  RMSE : 0.00001530
  MAE  : 0.00000904
  R2   : 0.99999405
  MAPE : 1.9124%


In [6]:
# ============================================================
# Method 2: Stacking
# RF, LR 예측값을 피처로 삼아 메타모델(LR) 학습
# ============================================================
# train 데이터에서 cross_val_predict로 OOF(Out-of-Fold) 예측 생성
# -> 과적합 방지를 위해 학습 데이터에 본 적 없는 예측값 사용
rf_oof = cross_val_predict(rf, X_train, y_train, cv=3)
lr_oof = cross_val_predict(lr, X_train, y_train, cv=3)

# 메타 피처 생성
meta_train = np.column_stack([rf_oof, lr_oof])
meta_test  = np.column_stack([rf_pred_test, lr_pred_test])

# 메타모델 학습 (Linear Regression)
meta_model = LinearRegression()
meta_model.fit(meta_train, y_train)

stack_pred = meta_model.predict(meta_test)

stack_rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
stack_mae  = mean_absolute_error(y_test, stack_pred)
stack_r2   = r2_score(y_test, stack_pred)
stack_mape = np.mean(np.abs((y_test - stack_pred) / y_test)) * 100

print(f"\n[Method 2: Stacking (meta: LinearRegression)]")
print(f"  meta_model coefficients: RF={meta_model.coef_[0]:.4f}, LR={meta_model.coef_[1]:.4f}")
print(f"  RMSE : {stack_rmse:.8f}")
print(f"  MAE  : {stack_mae:.8f}")
print(f"  R2   : {stack_r2:.8f}")
print(f"  MAPE : {stack_mape:.4f}%")


[Method 2: Stacking (meta: LinearRegression)]
  meta_model coefficients: RF=0.9469, LR=0.0531
  RMSE : 0.00001331
  MAE  : 0.00000786
  R2   : 0.99999550
  MAPE : 0.6280%


In [7]:
# ============================================================
# 단일 모델 결과와 비교
# ============================================================
def get_metrics(y_true, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae":  mean_absolute_error(y_true, y_pred),
        "r2":   r2_score(y_true, y_pred),
        "mape": np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    }

rf_metrics  = get_metrics(y_test, rf_pred_test)
lr_metrics  = get_metrics(y_test, lr_pred_test)

print(f"\n=== 최종 비교 ===")
results = pd.DataFrame([
    {"model": "RandomForest",      **rf_metrics},
    {"model": "LinearRegression",  **lr_metrics},
    {"model": "Weighted Blending", "rmse": blend_rmse, "mae": blend_mae, "r2": blend_r2, "mape": blend_mape},
    {"model": "Stacking",          "rmse": stack_rmse, "mae": stack_mae, "r2": stack_r2, "mape": stack_mape},
])
print(results.to_string(index=False))


=== 최종 비교 ===
            model     rmse      mae       r2      mape
     RandomForest 0.000014 0.000008 0.999995  0.066397
 LinearRegression 0.000061 0.000027 0.999904 10.757168
Weighted Blending 0.000015 0.000009 0.999994  1.912364
         Stacking 0.000013 0.000008 0.999995  0.628016


================= stacking 모델 저장 =======================

In [15]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'],
                               config['paths']['processed_file'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

In [17]:
# ============================================================
# 데이터 증강
# ============================================================
df_full = pd.read_parquet(PROCESSED_PATH)
print(f"Original rows: {len(df_full):,}")

# SAMPLE_SIZE = 5_000_000  # 500만 행 샘플링
SAMPLE_SIZE = 2_000_000  
df = df_full.sample(n=SAMPLE_SIZE, random_state=config['model']['random_state'])
del df_full  # 메모리 해제
import gc; gc.collect()
print(f"Sampled rows: {len(df):,}")

# ============================================================
# 데이터 증강 : duration 배수로 복제
# energy = power * duration / 3600 / 1000 이므로 duration에 선형
# ============================================================
DURATION_MULTIPLIERS = [1, 2, 4, 6, 12]  # 300~3600초 (5분~1시간)

augmented_dfs = []
for multiplier in DURATION_MULTIPLIERS:
    df_aug = df.copy()
    df_aug['duration'] = df_aug['duration'] * multiplier
    df_aug['energy_kwh'] = df_aug['energy_kwh'] * multiplier
    augmented_dfs.append(df_aug)

df_augmented = pd.concat(augmented_dfs, ignore_index=True)
del augmented_dfs; gc.collect()

print(f"Augmented rows: {len(df_augmented):,}")
print(f"Duration range: {df_augmented['duration'].min():.0f} ~ {df_augmented['duration'].max():.0f} sec")

Original rows: 19,523,808
Sampled rows: 2,000,000
Augmented rows: 10,000,000
Duration range: 1 ~ 3600 sec


In [18]:
df = pd.read_parquet(PROCESSED_PATH)

features = ["cpu", "memory", "duration"]
target   = "energy_kwh"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# ============================================================
# LR 먼저 학습 (고정)
# ============================================================
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_oof  = cross_val_predict(lr, X_train, y_train, cv=5)
lr_pred = lr.predict(X_test)

# ============================================================
# RF 하이퍼파라미터 조합 비교
# min_samples_split : 노드 분할 최소 샘플 수 (높을수록 덜 복잡한 트리)
# min_samples_leaf  : 리프 노드 최소 샘플 수 (높을수록 덜 복잡한 트리)
# max_depth         : 트리 최대 깊이
# ============================================================
rf_configs = [
    {"max_depth": 10, "min_samples_split": 10,  "min_samples_leaf": 5},
    {"max_depth": 15, "min_samples_split": 10,  "min_samples_leaf": 5},
    {"max_depth": 20, "min_samples_split": 10,  "min_samples_leaf": 5},
    {"max_depth": 20, "min_samples_split": 50,  "min_samples_leaf": 20},
    {"max_depth": 25, "min_samples_split": 10,  "min_samples_leaf": 5},
    {"max_depth": None, "min_samples_split": 50, "min_samples_leaf": 20},
]

print("\n=== 증강 데이터 기준 RF 하이퍼파라미터 비교 ===\n")
results = []
best_mape   = float('inf')
best_model  = None
best_lr     = lr
best_meta   = None
best_config = None

for cfg in rf_configs:
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=cfg["max_depth"],
        min_samples_split=cfg["min_samples_split"],
        min_samples_leaf=cfg["min_samples_leaf"],
        random_state=config['model']['random_state'],
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    rf_oof  = cross_val_predict(rf, X_train, y_train, cv=5)
    rf_pred = rf.predict(X_test)

    meta_train = np.column_stack([rf_oof, lr_oof])
    meta_test  = np.column_stack([rf_pred, lr_pred])

    meta = LinearRegression()
    meta.fit(meta_train, y_train)
    stack_pred = meta.predict(meta_test)

    rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
    mae  = mean_absolute_error(y_test, stack_pred)
    r2   = r2_score(y_test, stack_pred)
    mape = np.mean(np.abs((y_test - stack_pred) / y_test)) * 100

    label = f"depth={cfg['max_depth']} split={cfg['min_samples_split']} leaf={cfg['min_samples_leaf']}"
    results.append({"config": label, "rmse": rmse, "mae": mae, "r2": r2, "mape": mape,
                    "rf_coef": meta.coef_[0], "lr_coef": meta.coef_[1]})

    print(f"[{label}]")
    print(f"  RMSE={rmse:.8f} | R2={r2:.8f} | MAPE={mape:.4f}%")
    print(f"  meta coef -> RF={meta.coef_[0]:.4f}, LR={meta.coef_[1]:.4f}\n")

    # 최고 성능 추적 (MAPE 기준)
    if mape < best_mape:
        best_mape   = mape
        best_model  = rf
        best_meta   = meta
        best_config = cfg
        best_rmse   = rmse
        best_mae    = mae
        best_r2     = r2

print("\n=== 최종 비교 ===")
print(pd.DataFrame(results)[["config", "rmse", "r2", "mape"]].to_string(index=False))

print(f"\n=== 최적 config ===")
print(f"  {best_config}")
print(f"  MAPE: {best_mape:.4f}%")


Train: 15,619,046 | Test: 3,904,762

=== 증강 데이터 기준 RF 하이퍼파라미터 비교 ===

[depth=10 split=10 leaf=5]
  RMSE=0.00001335 | R2=0.99999547 | MAPE=0.6388%
  meta coef -> RF=0.9459, LR=0.0541

[depth=15 split=10 leaf=5]
  RMSE=0.00000322 | R2=0.99999974 | MAPE=0.0592%
  meta coef -> RF=0.9951, LR=0.0049

[depth=20 split=10 leaf=5]
  RMSE=0.00000265 | R2=0.99999982 | MAPE=0.0467%
  meta coef -> RF=0.9957, LR=0.0043

[depth=20 split=50 leaf=20]
  RMSE=0.00000421 | R2=0.99999955 | MAPE=0.0901%
  meta coef -> RF=0.9919, LR=0.0081

[depth=25 split=10 leaf=5]
  RMSE=0.00000264 | R2=0.99999982 | MAPE=0.0461%
  meta coef -> RF=0.9957, LR=0.0043

[depth=None split=50 leaf=20]
  RMSE=0.00000420 | R2=0.99999955 | MAPE=0.0896%
  meta coef -> RF=0.9919, LR=0.0081


=== 최종 비교 ===
                     config     rmse       r2     mape
   depth=10 split=10 leaf=5 0.000013 0.999995 0.638805
   depth=15 split=10 leaf=5 0.000003 1.000000 0.059223
   depth=20 split=10 leaf=5 0.000003 1.000000 0.046728
  depth=20 sp

In [19]:
import json
# ============================================================
# 최적 모델 저장
# ============================================================
rf_file   = os.path.join(MODEL_PATH, config['model']['model_names']['randomforest'])
lr_file   = os.path.join(MODEL_PATH, config['model']['model_names']['linearregression'])
meta_file = os.path.join(MODEL_PATH, config['model']['model_names']['stacking_meta'])

with open(rf_file, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nSaved RF   : {rf_file} ({os.path.getsize(rf_file)/(1024**2):.1f} MB)")

with open(lr_file, 'wb') as f:
    pickle.dump(best_lr, f)
print(f"Saved LR   : {lr_file} ({os.path.getsize(lr_file)/(1024**2):.1f} MB)")

with open(meta_file, 'wb') as f:
    pickle.dump(best_meta, f)
print(f"Saved Meta : {meta_file} ({os.path.getsize(meta_file)/(1024**2):.1f} MB)")

# ============================================================
# phase1_full_results.json 업데이트
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json'])
with open(result_file, 'r') as f:
    results_json = json.load(f)

results_json['stacking'] = {
    'method'      : 'Stacking (RF + LR) with Data Augmentation',
    'base_models' : ['RandomForest', 'LinearRegression'],
    'meta_model'  : 'LinearRegression',
    'rf_params'   : {
        'n_estimators'     : 100,
        'max_depth'        : best_config['max_depth'],
        'min_samples_split': best_config['min_samples_split'],
        'min_samples_leaf' : best_config['min_samples_leaf'],
    },
    'augmentation': {
        'duration_multipliers': DURATION_MULTIPLIERS,
        'sample_size'         : SAMPLE_SIZE,
        'augmented_rows'      : SAMPLE_SIZE * len(DURATION_MULTIPLIERS),
    },
    'rf_coef'     : float(best_meta.coef_[0]),
    'lr_coef'     : float(best_meta.coef_[1]),
    'features'    : features,
    'rmse'        : float(best_rmse),
    'mae'         : float(best_mae),
    'r2'          : float(best_r2),
    'mape'        : float(best_mape),
}
results_json['best_model'] = 'Stacking'

with open(result_file, 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f"\nResults updated: {result_file}")
print("Done!")


Saved RF   : /content/drive/MyDrive/EcoTracing/models/energy_model_randomforest.pkl (6000.0 MB)
Saved LR   : /content/drive/MyDrive/EcoTracing/models/energy_model_linearregression.pkl (0.0 MB)
Saved Meta : /content/drive/MyDrive/EcoTracing/models/energy_model_stacking_meta.pkl (0.0 MB)

Results updated: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results.json
Done!


===================== 베스트 하이퍼파라미터 value 사용 모델 학습 ============================

In [8]:
features = ["cpu", "memory", "duration"]
target   = "energy_kwh"

X = df_augmented[features]
y = df_augmented[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

del df_augmented; gc.collect()
print(f"\nTrain: {len(X_train):,} | Test: {len(X_test):,}")


Train: 20,000,000 | Test: 5,000,000


In [9]:
# ============================================================
# 모델 학습
# ============================================================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
lr = LinearRegression()

rf.fit(X_train, y_train)
lr.fit(X_train, y_train)
print("Base models trained!")

Base models trained!


In [10]:
# ============================================================
# Stacking - OOF 예측으로 메타모델 학습
# ============================================================
rf_oof = cross_val_predict(rf, X_train, y_train, cv=5)
lr_oof = cross_val_predict(lr, X_train, y_train, cv=5)

meta_train = np.column_stack([rf_oof, lr_oof])

meta_model = LinearRegression()
meta_model.fit(meta_train, y_train)
print("Meta model trained!")
print(f"  RF coef : {meta_model.coef_[0]:.4f}")
print(f"  LR coef : {meta_model.coef_[1]:.4f}")


Meta model trained!
  RF coef : 0.9986
  LR coef : 0.0014


In [11]:
# ============================================================
# 최종 평가
# ============================================================
rf_pred = rf.predict(X_test)
lr_pred = lr.predict(X_test)
meta_test = np.column_stack([rf_pred, lr_pred])
stack_pred = meta_model.predict(meta_test)

rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
mae = mean_absolute_error(y_test, stack_pred)
r2 = r2_score(y_test, stack_pred)
mape = np.mean(np.abs((y_test - stack_pred) / y_test)) * 100

print(f"\n[Stacking 최종 성능 (데이터 증강 후)]")
print(f" RMSE : {rmse:.8f}")
print(f" MAE: {mae:.8f}")
print(f" R2 : {r2:.8f}")
print(f" MAPE : {mape:.4f}%")


[Stacking 최종 성능 (데이터 증강 후)]
 RMSE : 0.00003740
 MAE: 0.00001589
 R2 : 0.99999968
 MAPE : 0.0908%


In [12]:
# ============================================================
# 모델 저장 (config 기반)
# ============================================================

rf_file   = os.path.join(MODEL_PATH, config['model']['model_names']['randomforest'])
lr_file   = os.path.join(MODEL_PATH, config['model']['model_names']['linearregression'])
meta_file = os.path.join(MODEL_PATH, config['model']['model_names']['stacking_meta'])

with open(rf_file, 'wb') as f:
    pickle.dump(rf, f)
print(f"\nSaved RF   : {rf_file} ({os.path.getsize(rf_file)/(1024**2):.1f} MB)")

with open(lr_file, 'wb') as f:
    pickle.dump(lr, f)
print(f"Saved LR   : {lr_file} ({os.path.getsize(lr_file)/(1024**2):.1f} MB)")

with open(meta_file, 'wb') as f:
    pickle.dump(meta_model, f)
print(f"Saved Meta : {meta_file} ({os.path.getsize(meta_file)/(1024**2):.1f} MB)")


Saved RF   : /content/drive/MyDrive/EcoTracing/models/energy_model_randomforest.pkl (298.4 MB)
Saved LR   : /content/drive/MyDrive/EcoTracing/models/energy_model_linearregression.pkl (0.0 MB)
Saved Meta : /content/drive/MyDrive/EcoTracing/models/energy_model_stacking_meta.pkl (0.0 MB)


In [14]:
# ============================================================
# phase1_full_results.json 업데이트
# ============================================================
import json

result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json'])
with open(result_file, 'r') as f:
    results = json.load(f)

results['stacking'] = {
    'method'      : 'Stacking (RF + LR) with Data Augmentation',
    'base_models' : ['RandomForest', 'LinearRegression'],
    'meta_model'  : 'LinearRegression',
    'rf_params'   : {
        'n_estimators'     : 100,
        'max_depth'        : 15,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
    },
    'augmentation': {
        'duration_multipliers': DURATION_MULTIPLIERS,
        'sample_size'         : SAMPLE_SIZE,
        'augmented_rows'      : SAMPLE_SIZE * len(DURATION_MULTIPLIERS),
    },
    'rf_coef'     : float(meta_model.coef_[0]),
    'lr_coef'     : float(meta_model.coef_[1]),
    'features'    : features,
    'rmse'        : float(rmse),
    'mae'         : float(mae),
    'r2'          : float(r2),
    'mape'        : float(mape),
}
results['best_model'] = 'Stacking'

with open(result_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nResults updated: {result_file}")


Results updated: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results.json
